# Stage 03a — Aircraft Identity, Scope & Maturity

**What it does.** For every patent in a batch, works out *which real aircraft it
relates to, what the patent is actually about, and how far along it is* — and
writes one workbook per batch:

    <data_matched>/<Batch_NN>/aircraft_identity_<Batch_NN>.xlsx

**Inputs:** `batches.xlsx` + the PatSeer export. No images, no figure crops — so
this runs independently of 00a/00b/01a and writes only its own file.

**The one thing to remember:** filter `aircraft_link == "Depicted"` before any
statistic grouped by aircraft. A `CompanyAttributed` row means "Joby filed this
and Joby makes the S4", not "this patent shows an S4".

| Question | Key columns |
|---|---|
| What is the patent about, at what level? | `scope`, `innovation_field` |
| Which architecture(s)? | `architecture_primary`, `architecture_all`, `architecture_count`, `architecture_pure` |
| Tied to one real aircraft? | `specificity`, `aircraft_link` |
| Which aircraft? | `aircraft_name` + `_source` + `_confidence` |
| Electric? | `is_electric`, `powertrain` |
| Its numbers? | `pax`, `mtow_kg`, `range_km`, …, `blades_primary`, `blades_all` |
| Where / what for? | `assignee_country`, `region`, `pub_office`, `industry_primary` |
| Accepted or only filed? | `legal_stage`, `maturity_tier`, `impact_tier`, citations |

Every field carries its own `*_source` and `*_confidence`. **Nothing is ever
guessed** — a patent with nothing known keeps empty cells and `needs_review`.

**How to run:** set `BATCH_ID` below, then run every cell top to bottom. The LLM
step is optional and off the critical path (cells 6–7).

All the logic lives in `src/` — see `src/identity_pipeline.py` for the runner
and `notebooks/README.md` for how this fits with the other notebooks.


In [ ]:
# ── 1. Configuration — the only knobs ──────────────────────────────────────
BATCH_ID = 1        # which Batch_NN sheet of batches.xlsx to process
LIMIT    = None     # e.g. 5 for a quick smoke test; None = the whole batch
USE_SBERT = True    # False = keyword/regex/gazetteer only (fast, no GPU needed)

# LLM step: "export" writes questions into the workbook for you to paste into a
# chat (no API key); "api" calls Claude directly; "off" skips it entirely.
LLM_MODE  = "export"
LLM_MODEL = "claude-opus-5"
LLM_ONLY_UNRESOLVED = True   # only ask about patents the gazetteer could not name

In [ ]:
# ── 2. Setup: repo on sys.path, config, models ─────────────────────────────
import sys
from pathlib import Path

_cwd = Path().resolve()
repo_root = next((c for c in [_cwd, *_cwd.parents]
                  if (c / "src").exists() and (c / "config.yaml").exists()), None)
if repo_root is None:
    raise RuntimeError(f"Cannot find repo root from {_cwd}. Run from inside Patent-Labelling-Tools.")
for p in [str(repo_root), str(repo_root / "src")]:
    while p in sys.path:
        sys.path.remove(p)
sys.path.insert(0, str(repo_root / "src"))
sys.path.insert(0, str(repo_root))

import pandas as pd
from src.config_loader import load_config
from src import identity_pipeline as pipeline
from src import aircraft_identity as ai

cfg = load_config()
print(f"repo_root : {repo_root}")

In [ ]:
# ── 3. Load PatentSBERTa (optional) ────────────────────────────────────────
# Same model and cache folder as 01a_wizard_feed, so the weights are not
# downloaded twice. A missing GPU or an offline machine degrades this stage to
# its keyword/regex/gazetteer passes rather than failing the run.
import os

sbert = None
if USE_SBERT:
    # Must be set BEFORE sentence_transformers imports huggingface_hub, which
    # freezes the cache path at import time (same note 01a carries).
    os.environ.setdefault("HF_HUB_CACHE", str(cfg["paths"]["sbert_cache"]))
    os.environ.setdefault("HF_HOME",      str(cfg["paths"]["sbert_cache"]))
    try:
        import torch
        from sentence_transformers import SentenceTransformer
        device = "cuda" if torch.cuda.is_available() else "cpu"
        sbert = SentenceTransformer("AI-Growth-Lab/PatentSBERTa",
                                    cache_folder=str(cfg["paths"]["sbert_cache"]),
                                    device=device)
        print(f"PatentSBERTa loaded on {device}.")
    except Exception as exc:
        print(f"⚠  SBERT unavailable ({type(exc).__name__}: {exc}) — "
              f"running keyword/regex/gazetteer only.")
else:
    print("USE_SBERT = False — running keyword/regex/gazetteer only.")

In [ ]:
# ── 4. Read the batch, the PatSeer export and the gazetteer ────────────────
inputs = pipeline.load_inputs(cfg, BATCH_ID, limit=LIMIT, repo_root=repo_root)

In [ ]:
# ── 5. Run every signal and assemble the rows ──────────────────────────────
# Order inside: identity row -> scope/specificity -> maturity -> percentiles.
# See src/identity_pipeline.py for why that order is not interchangeable.
results = pipeline.analyse(inputs, sbert)

In [ ]:
# ── 6. Build the LLM questions (optional) ──────────────────────────────────
prompts, llm_answers = {}, {}

if LLM_MODE != "off":
    prompts = pipeline.build_prompts(inputs, results, only_unresolved=LLM_ONLY_UNRESOLVED)

if LLM_MODE == "api" and prompts:
    print(f"Calling {LLM_MODEL} for {len(prompts)} patent(s)...")
    llm_answers = ai.ask_claude(prompts, model=LLM_MODEL)
    results = pipeline.analyse(inputs, sbert, llm_answers=llm_answers)
elif LLM_MODE == "export":
    print(f"{len(prompts)} prompt(s) will be written to the LLM_Prompts sheet.\n"
          f"Paste each reply into its `llm_answer` cell, save, then run cell 8.\n"
          f"\nGive the chat this system prompt once, before the questions:\n")
    print(ai.LLM_SYSTEM_PROMPT)
else:
    print("LLM step skipped.")

In [ ]:
# ── 7. Write the workbook ──────────────────────────────────────────────────
# Backs up any existing file, then carries your hand edits forward: the notes /
# reviewer / pasted-answer columns always, plus any field whose *_source you set
# to "human". Safe to re-run over a sheet you have been editing.
out_path = pipeline.export(inputs, results, prompts)

In [ ]:
# ── 8. Merge pasted LLM answers back in (run after filling the sheet) ──────
INGEST_LLM_ANSWERS = False   # flip to True once the llm_answer column is filled

if INGEST_LLM_ANSWERS:
    # The whole analysis is redone, not patched: an LLM-sourced name is evidence
    # the patent depicts the aircraft in a way a gazetteer name is not, so
    # specificity and aircraft_link can legitimately change once answers land.
    llm_answers = pipeline.read_pasted_answers(out_path)
    results = pipeline.analyse(inputs, sbert, llm_answers=llm_answers)
    pipeline.export(inputs, results, prompts)
else:
    print("INGEST_LLM_ANSWERS = False — nothing to ingest yet.")

In [ ]:
# ── 9. Coverage report — what this batch can support in the thesis ─────────
ident = pipeline.report(results)

In [ ]:
# ── 10. Every batch at once (optional) ─────────────────────────────────────
# One workbook per batch plus a combined table. The LLM step is skipped here on
# purpose — see run_all_batches() for why.
RUN_ALL_BATCHES = False

if RUN_ALL_BATCHES:
    combined = pipeline.run_all_batches(cfg, sbert, repo_root=repo_root)
else:
    print("RUN_ALL_BATCHES = False — single-batch run only.")

## Where this leaves you

The workbook is the deliverable. **Identity** joins onto anything keyed by
`patent_id`; **Figures** says what each drawing depicts; **Evidence** is the
audit trail behind every value; **README** is the column dictionary.

### Three rules for using it in the thesis

1. **Filter `aircraft_link == "Depicted"`** before any per-aircraft statistic.
   The `CompanyAttributed` rows are genuine evidence about the *company* — just
   not about that aircraft's design.
2. **Rank maturity on `forward_citations_per_year`**, never the raw count. A
   2015 patent has had a decade to be cited and a 2023 one has not, and eVTOL
   filing volume rose steeply over that window, so raw counts sort the corpus by
   age and call it impact.
3. **Report the coverage, don't hide it.** "We identified the aircraft for N of
   M patents, and here is the source breakdown" is a stronger result than a
   table that quietly implies all M were identified. The naming rate is a
   property of patent drafting practice, not of this method.

### To improve coverage, in order of payoff

1. **Grow `reference/evtol_gazetteer.csv`** — the only source precise enough to
   carry a spec number into the thesis. The load cell prints how many patents it
   reaches by company; each company you add converts a whole cluster at once.
   `company_canonical` must match a name from `COMPANY_LOOKUP` in
   `src/grouper.py` or the row will never match.
2. **Fill its numeric columns** from type certificates or company datasheets,
   recording each source in `spec_source`. Everything ships blank on purpose.
3. **Run the LLM step**, then spot-check it against the rows the gazetteer also
   covers. That overlap measures how far to trust it, and belongs in the
   methodology chapter.
4. **Calibrate the scope pass against the wizard.** It already asks a human for
   `archCount` and `notPureArch` — comparing those to the predicted
   `architecture_count` / `architecture_pure` is a free accuracy figure on your
   own data.
5. **Correct the sheet by hand** where you know better, setting that field's
   `*_source` to `human`. Those survive every re-run.
